# 🌳 How Decision Trees Are Built: A Step-by-Step Guide  
### *With Interactive Demo in Google Colab*

## 🔍 What Is a Decision Tree?

A **decision tree** is a supervised machine learning algorithm used for **classification** (e.g., "Will this loan be approved?") and **regression** (e.g., "What will the house price be?").  

It works by **recursively splitting** the data into smaller subsets based on feature values, aiming to create **pure nodes**—where all samples belong to the same class (for classification) or have similar values (for regression).

The result is a **tree-like structure**:
- **Root node**: Entire dataset  
- **Internal nodes**: Feature-based decisions (e.g., "Is Credit Score ≥ 700?")  
- **Leaf nodes**: Final predictions (e.g., "Approve = Yes")

## 🧠 How Is the Tree Built? The Core Idea

The algorithm builds the tree **top-down**, using a **greedy strategy**:

> At each node, choose the **best possible split** that **maximizes purity** in the child nodes.

This process is **recursive**: once a split is made, the same logic is applied to each child node—until stopping conditions are met.

## ⚙️ Key Concepts Behind Splitting

### 1. **Splitting Criterion: Gini Impurity (for Classification)**

The most common metric in algorithms like **CART** (used in `scikit-learn`) is **Gini Impurity**.

#### 🔢 What Is Gini Impurity?

Gini measures how **mixed** a node is.  
- **Gini = 0** → perfectly pure (all samples same class)  
- **Gini = 0.5** → maximum impurity (equal mix of classes)

#### 📐 Formula:
For a node with classes $ C_1, C_2, ..., C_k $, and $ p_i $ = proportion of class $ i $:

$$
\text{Gini} = 1 - \sum_{i=1}^{k} p_i^2
$$

#### ✅ Example:
- Node has 6 samples: 4 "Yes", 2 "No"  
- $ p_{\text{Yes}} = 4/6 = 0.67 $, $ p_{\text{No}} = 2/6 = 0.33 $  
- $ \text{Gini} = 1 - (0.67^2 + 0.33^2) = 1 - (0.4489 + 0.1089) = 1 - 0.5578 = \textbf{0.4422} $

### 2. **How Is the Best Split Chosen?**

The algorithm **evaluates every possible split** and picks the one that **reduces impurity the most**.

#### 🔍 For Numerical Features (e.g., Income, Credit Score)

1. **Sort** all unique values of the feature  
2. Compute **midpoints** between consecutive values → these are **candidate thresholds**  
3. For each threshold:
   - Split data into **left** (feature < threshold) and **right** (feature ≥ threshold)
   - Compute **weighted Gini** of children:
   $$
     \text{Weighted Gini} = \frac{N_{\text{left}}}{N_{\text{total}}} \cdot \text{Gini}_{\text{left}} + \frac{N_{\text{right}}}{N_{\text{total}}} \cdot \text{Gini}_{\text{right}}
  $$
   - Compute **Gini Gain**:
  $$
     \text{Gain} = \text{Gini}_{\text{parent}} - \text{Weighted Gini}
  $$
4. Choose the threshold with **highest Gini Gain**

> 💡 **Why midpoints?**  
> Any value between two data points gives the same split. The midpoint is a safe, unambiguous choice.

#### 🔤 For Categorical Features (e.g., Employed: Yes/No)

- Only **one meaningful split** exists for binary features: {Yes} vs {No}  
- For multi-class features, test all possible groupings (though `scikit-learn` requires one-hot encoding)

## 📊 Concrete Example: Loan Approval Dataset

Let’s say we have this dataset:

| Income ($k) | Credit Score | Employed | Approve |
|-------------|--------------|----------|---------|
| 40          | 620          | No       | No      |
| 60          | 650          | Yes      | Yes     |
| 35          | 580          | No       | No      |
| 80          | 720          | Yes      | Yes     |
| 45          | 690          | Yes      | Yes     |
| 50          | 600          | No       | No      |
| 90          | 750          | Yes      | Yes     |
| 55          | 670          | Yes      | No      |

### Step 1: Root Node
- Samples: 8  
- Labels: [No, Yes, No, Yes, Yes, No, Yes, No] → 5 Yes, 3 No  
- Gini = $ 1 - (5/8)^2 - (3/8)^2 = 1 - 0.3906 - 0.1406 = \textbf{0.4688} $

### Step 2: Evaluate Splits

#### Try: `Credit Score ≥ 680?`
- Left (<680): [620, 650, 580, 600, 670] → Labels: [No, Yes, No, No, No] → 1 Yes, 4 No  
  - Gini = $ 1 - (0.2^2 + 0.8^2) = 0.32 $
- Right (≥680): [720, 690, 750] → Labels: [Yes, Yes, Yes] → Gini = 0  
- Weighted Gini = $ (5/8)(0.32) + (3/8)(0) = 0.20 $  
- **Gini Gain = 0.4688 – 0.20 = 0.2688**

#### Try: `Employed = Yes?`
- Left (No): [No, No, No] → Gini = 0  
- Right (Yes): [Yes, Yes, Yes, Yes, No] → Gini = $ 1 - (0.8^2 + 0.2^2) = 0.32 $  
- Weighted Gini = $ (3/8)(0) + (5/8)(0.32) = 0.20 $  
- **Gini Gain = 0.2688** ← Same!

Suppose the algorithm picks `Employed` as the root split.

### Step 3: Recurse on Child Nodes
- Left child (Employed=No): all "No" → **leaf node**  
- Right child (Employed=Yes): not pure → split again (e.g., on `Credit Score`)

This continues until all leaves are pure or stopping conditions are met.

## 🛑 When Does Splitting Stop?

Common stopping criteria:
- **Maximum depth** reached (e.g., depth = 3)  
- **Minimum samples per leaf** (e.g., ≥2 samples)  
- **Node is pure** (Gini = 0)  
- **No split improves impurity** (Gain ≤ 0)

## 🌐 How Prediction Works

To predict a new sample:
1. Start at the **root**
2. At each internal node, **follow the branch** based on the feature value
3. When you reach a **leaf**, return its **majority class** (or mean value for regression)

> Example:  
> Input: Income=70, Credit=710, Employed=Yes  
> Path: Employed=Yes → Credit ≥ 680 → Leaf → **Approve = Yes**

## 🧪 Interactive Demo in Google Colab

Now, let’s **see this in action**! The cells below will:

1. Let you **choose sample size**  
2. **Generate random loan data**  
3. **Build the tree step-by-step** (showing Gini, thresholds, and gains)  
4. **Visualize the tree** after each split  
5. Let you **make predictions** and **see the decision path**

> ✅ **Run the cells in order** to explore how decision trees really work!

In [ ]:
# @title
# ----------------------------
# Interactive Decision Tree Demo with Custom Sample Size
# ----------------------------

# Install & Import
!apt-get install -y graphviz
!pip install graphviz

import pandas as pd
import numpy as np
from graphviz import Digraph
from IPython.display import display, clear_output
import ipywidgets as widgets
from collections import deque

# --- UI Controls ---
sample_size_slider = widgets.IntSlider(
    value=10,
    min=5,
    max=100,
    step=1,
    description='Sample Size:',
    style={'description_width': 'initial'}
)

start_button = widgets.Button(
    description="Generate Data & Start Demo",
    button_style='primary',
    icon='play'
)

ui_controls = widgets.VBox([sample_size_slider, start_button])

# Global state (will be reset on start)
tree_nodes = {}
edges = []
queue = deque()
output_area = widgets.Output()
next_button = widgets.Button(description="Next Step", disabled=True)
demo_ui = None

# Helper: Generate realistic synthetic data
def generate_loan_data(n_samples):
    np.random.seed(42)  # for reproducibility during demo
    income = np.random.normal(60, 20, n_samples).clip(20, 120).astype(int)
    credit = np.random.normal(680, 60, n_samples).clip(500, 800).astype(int)
    employed = np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7])

    # Realistic approval logic (with noise)
    approve_prob = (
        (income > 50) * 0.3 +
        (credit > 650) * 0.4 +
        (employed == 1) * 0.3
    )
    approve_prob = np.clip(approve_prob + np.random.normal(0, 0.1, n_samples), 0, 1)
    approve = (np.random.rand(n_samples) < approve_prob).astype(int)

    df = pd.DataFrame({
        'Income': income,
        'Credit_Score': credit,
        'Employed': employed,
        'Approve': approve
    })
    return df

# Helper: Gini impurity
def gini(y):
    if len(y) == 0: return 0
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - np.sum(p**2)

# Helper: Evaluate all splits
def evaluate_all_splits(X_node, y_node):
    parent_gini = gini(y_node)
    n = len(y_node)
    candidates = []
    for feat in X_node.columns:
        if feat == 'Employed':
            left_mask = X_node[feat] == 0
            right_mask = ~left_mask
            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue
            w_gini = (left_mask.sum()/n)*gini(y_node[left_mask]) + (right_mask.sum()/n)*gini(y_node[right_mask])
            gain = parent_gini - w_gini
            candidates.append({
                'feature': feat,
                'threshold': 'Yes/No',
                'gain': gain,
                'left_mask': left_mask,
                'right_mask': right_mask,
                'type': 'categorical'
            })
        else:
            vals = sorted(X_node[feat].unique())
            for i in range(len(vals)-1):
                thr = (vals[i] + vals[i+1]) / 2
                left_mask = X_node[feat] < thr
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                w_gini = (left_mask.sum()/n)*gini(y_node[left_mask]) + (right_mask.sum()/n)*gini(y_node[right_mask])
                gain = parent_gini - w_gini
                candidates.append({
                    'feature': feat,
                    'threshold': thr,
                    'gain': gain,
                    'left_mask': left_mask,
                    'right_mask': right_mask,
                    'type': 'numerical'
                })
    candidates.sort(key=lambda x: x['gain'], reverse=True)
    return candidates

# Helper: Visualize tree
def visualize_tree():
    dot = Digraph(comment='Decision Tree')
    dot.attr(rankdir='TB', size='12')
    for nid, ndata in tree_nodes.items():
        if ndata['is_leaf']:
            pred = 'Yes' if ndata['pred'] == 1 else 'No'
            label = f"Leaf\nPred: {pred}\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
            color = '#e5f5e0' if ndata['pred'] == 1 else '#fee0d2'
            dot.node(str(nid), label, style='filled', fillcolor=color)
        else:
            if ndata['feature'] is None:
                label = f"Root\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
            else:
                if ndata['feature'] == 'Employed':
                    label = f"Employed?\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
                else:
                    label = f"{ndata['feature']} < {ndata['threshold']:.1f}?\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
            dot.node(str(nid), label)
    for parent, child, lbl in edges:
        dot.edge(str(parent), str(child), label=lbl)
    return dot

# Button: Start Demo
def on_start_click(b):
    global tree_nodes, edges, queue, demo_ui, df_global

    n_samples = sample_size_slider.value
    df = generate_loan_data(n_samples)
    df_global = df  # save for reference

    with output_area:
        clear_output()
        print(f"✅ Generated {n_samples} samples:")
        display(df.head(10))  # show first 10
        if n_samples > 10:
            print(f"... and {n_samples - 10} more rows.")
        print("\n" + "="*60)

    # Reset tree state
    tree_nodes.clear()
    edges.clear()
    queue.clear()

    X = df[['Income', 'Credit_Score', 'Employed']]
    y = df['Approve']

    # Initialize root
    root_gini = gini(y)
    tree_nodes[0] = {
        'is_leaf': False,
        'feature': None,
        'threshold': None,
        'gini': root_gini,
        'samples': len(y),
        'indices': list(X.index),
        'X': X.copy(),
        'y': y.copy(),
        'depth': 0
    }
    queue.append(0)

    # Enable Next Step
    next_button.disabled = False

    # Show initial tree (just root)
    with output_area:
        dot = visualize_tree()
        display(dot)

# Button: Next Step
def on_next_click(b):
    global queue

    if not queue:
        with output_area:
            clear_output()
            print("🎉 Tree construction complete!")
            dot = visualize_tree()
            display(dot)
        next_button.disabled = True
        return

    node_id = queue.popleft()
    node = tree_nodes[node_id]
    X_node, y_node, depth = node['X'], node['y'], node['depth']

    with output_area:
        clear_output()

        # Stopping conditions
        if depth >= 4 or len(y_node) < 2 or gini(y_node) == 0:
            pred = int(y_node.mode()[0]) if len(y_node) > 0 else 0
            tree_nodes[node_id].update({'is_leaf': True, 'pred': pred, 'gini': gini(y_node)})
            print(f"✅ Created LEAF node (ID={node_id})")
            dot = visualize_tree()
            display(dot)
            return

        # Evaluate splits
        candidates = evaluate_all_splits(X_node, y_node)
        if not candidates or candidates[0]['gain'] <= 0:
            pred = int(y_node.mode()[0])
            tree_nodes[node_id].update({'is_leaf': True, 'pred': pred, 'gini': gini(y_node)})
            print(f"⚠️ No useful split → LEAF (ID={node_id})")
            dot = visualize_tree()
            display(dot)
            return

        best = candidates[0]
        tree_nodes[node_id].update({
            'feature': best['feature'],
            'threshold': best['threshold'] if best['type'] == 'numerical' else None,
            'is_leaf': False
        })

        # Create children
        global node_id_counter
        if not hasattr(on_next_click, 'node_id_counter'):
            on_next_click.node_id_counter = 1
        left_id = on_next_click.node_id_counter
        on_next_click.node_id_counter += 1
        right_id = on_next_click.node_id_counter
        on_next_click.node_id_counter += 1

        X_left = X_node[best['left_mask']]
        y_left = y_node[best['left_mask']]
        X_right = X_node[best['right_mask']]
        y_right = y_node[best['right_mask']]

        tree_nodes[left_id] = {
            'is_leaf': False, 'feature': None, 'threshold': None,
            'gini': gini(y_left), 'samples': len(y_left),
            'indices': list(X_left.index), 'X': X_left.copy(), 'y': y_left.copy(),
            'depth': depth + 1
        }
        tree_nodes[right_id] = {
            'is_leaf': False, 'feature': None, 'threshold': None,
            'gini': gini(y_right), 'samples': len(y_right),
            'indices': list(X_right.index), 'X': X_right.copy(), 'y': y_right.copy(),
            'depth': depth + 1
        }

        edges.append((node_id, left_id, 'No' if best['feature'] == 'Employed' else 'True'))
        edges.append((node_id, right_id, 'Yes' if best['feature'] == 'Employed' else 'False'))

        queue.append(left_id)
        queue.append(right_id)

        # Explain split
        print(f"✂️ Evaluating splits for node (ID={node_id})")
        print(f"   Samples: {len(y_node)} | Gini: {gini(y_node):.4f}\n")

        if best['feature'] != 'Employed':
            print(f"🔍 Top candidate thresholds for '{best['feature']}':")
            feat_cands = [c for c in candidates if c['feature'] == best['feature']][:5]
            for c in feat_cands:
                print(f"   {best['feature']} < {c['threshold']:6.1f} → Gain = {c['gain']:.4f}")
            print(f"\n✅ BEST: {best['feature']} < {best['threshold']:.1f} (Gain = {best['gain']:.4f})")
        else:
            print(f"✅ BEST SPLIT: Employed = No vs Yes (Gain = {best['gain']:.4f})")

        print(f"\n➡️ Created children: Left (ID={left_id}), Right (ID={right_id})")
        dot = visualize_tree()
        display(dot)

start_button.on_click(on_start_click)
next_button.on_click(on_next_click)

# Assemble full UI
demo_ui = widgets.VBox([
    ui_controls,
    widgets.HBox([next_button]),
    output_area
])

display(demo_ui)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
# @title
# ----------------------------
# Cell: Make Prediction & Visualize Decision Path
# ----------------------------

from IPython.display import display, Markdown
import ipywidgets as widgets

# Reconstruct the final sklearn-like tree from our manual build for prediction
# We'll use the global tree_nodes built in the previous demo

def predict_with_path(income, credit, employed):
    """Traverse the tree and return prediction + path of node IDs."""
    path = []
    current_id = 0  # start at root

    while not tree_nodes[current_id]['is_leaf']:
        path.append(current_id)
        node = tree_nodes[current_id]
        feat = node['feature']

        if feat == 'Employed':
            if employed == 0:
                # Go to "No" child (first edge)
                next_id = edges[[i for i, (p, c, l) in enumerate(edges) if p == current_id and l == 'No'][0]][1]
            else:
                next_id = edges[[i for i, (p, c, l) in enumerate(edges) if p == current_id and l == 'Yes'][0]][1]
        else:
            threshold = node['threshold']
            if feat == 'Income':
                val = income
            elif feat == 'Credit_Score':
                val = credit
            else:
                val = employed

            if val < threshold:
                # Go to "True" child
                next_id = edges[[i for i, (p, c, l) in enumerate(edges) if p == current_id and l == 'True'][0]][1]
            else:
                next_id = edges[[i for i, (p, c, l) in enumerate(edges) if p == current_id and l == 'False'][0]][1]

        current_id = next_id

    path.append(current_id)  # include leaf
    pred = tree_nodes[current_id]['pred']
    return pred, path

def visualize_path_highlighted(path):
    """Render tree with path highlighted."""
    dot = Digraph(comment='Decision Path')
    dot.attr(rankdir='TB', size='12')

    for nid, ndata in tree_nodes.items():
        if ndata['is_leaf']:
            pred = 'Yes' if ndata['pred'] == 1 else 'No'
            label = f"Leaf\nPred: {pred}\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
            if nid in path:
                color = '#c7e9c0' if ndata['pred'] == 1 else '#fdae6b'  # brighter if in path
                dot.node(str(nid), label, style='filled,bold', fillcolor=color, penwidth='3')
            else:
                color = '#e5f5e0' if ndata['pred'] == 1 else '#fee0d2'
                dot.node(str(nid), label, style='filled', fillcolor=color)
        else:
            if ndata['feature'] is None:
                label = f"Root\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
            else:
                if ndata['feature'] == 'Employed':
                    label = f"Employed?\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"
                else:
                    label = f"{ndata['feature']} < {ndata['threshold']:.1f}?\nSamples: {ndata['samples']}\nGini: {ndata['gini']:.3f}"

            if nid in path:
                dot.node(str(nid), label, style='bold', penwidth='3', color='blue')
            else:
                dot.node(str(nid), label)

    for parent, child, lbl in edges:
        if parent in path and child in path:
            dot.edge(str(parent), str(child), label=lbl, color='blue', penwidth='2')
        else:
            dot.edge(str(parent), str(child), label=lbl)

    return dot

# UI Widgets
income_widget = widgets.IntSlider(value=60, min=20, max=120, step=5, description='Income ($k):')
credit_widget = widgets.IntSlider(value=700, min=500, max=800, step=10, description='Credit Score:')
employed_widget = widgets.Dropdown(options=[('No', 0), ('Yes', 1)], value=1, description='Employed:')

predict_button = widgets.Button(description="Predict Approval", button_style='success')
output_pred = widgets.Output()

def on_predict_click(b):
    income = income_widget.value
    credit = credit_widget.value
    employed = employed_widget.value

    try:
        pred, path = predict_with_path(income, credit, employed)
        result = "✅ Approved" if pred == 1 else "❌ Denied"

        with output_pred:
            output_pred.clear_output()
            print(f"\n🔍 Input: Income=${income}k, Credit={credit}, Employed={'Yes' if employed else 'No'}")
            print(f"🎯 Prediction: {result}\n")

            print("📜 Decision Path:")
            for i, node_id in enumerate(path):
                node = tree_nodes[node_id]
                if node['is_leaf']:
                    print(f"  Step {i+1}: Leaf → Predict {'Yes' if node['pred'] == 1 else 'No'}")
                else:
                    if node['feature'] == 'Employed':
                        direction = "No" if employed == 0 else "Yes"
                        print(f"  Step {i+1}: Check Employed? → Go to '{direction}' branch")
                    else:
                        val = income if node['feature'] == 'Income' else credit
                        direction = "True" if val < node['threshold'] else "False"
                        print(f"  Step {i+1}: Check {node['feature']} < {node['threshold']:.1f}? (Value={val}) → Go to '{'Left' if direction=='True' else 'Right'}' branch")

            print("\n🎨 Visualizing decision path...")
            dot = visualize_path_highlighted(path)
            display(dot)

    except Exception as e:
        with output_pred:
            print("⚠️ Error: Tree not built yet. Please complete the tree-building demo first.")

predict_button.on_click(on_predict_click)

# Assemble UI
pred_ui = widgets.VBox([
    widgets.HTML("<h3>🔍 Make a Prediction</h3>"),
    income_widget,
    credit_widget,
    employed_widget,
    predict_button,
    output_pred
])

display(pred_ui)

## 💡 Key Takeaways

| Concept | Explanation |
|--------|-------------|
| **Gini Impurity** | Measures node purity: lower = better |
| **Split Selection** | Greedily pick split with highest Gini Gain |
| **Numerical Splits** | Test midpoints between sorted unique values |
| **Categorical Splits** | Binary: one split; Multi-class: test groupings |
| **Tree Growth** | Recursive, top-down, stops when pure or criteria met |
| **Prediction** | Traverse tree based on feature values → return leaf prediction |

## 📚 Further Reading

- **Gini vs. Entropy**: Both work similarly; Gini is faster, Entropy more sensitive to balanced splits  
- **Overfitting**: Deep trees memorize noise → use **pruning**, **max_depth**, or **ensembles** (Random Forest, XGBoost)  
- **Regression Trees**: Use **MSE** instead of Gini; predict **mean** of leaf

## ✅ Try It Yourself!

Use the Colab demo to:
- See **how thresholds are derived**  
- Watch **Gini decrease** with each split  
- Understand **why a tree makes a certain prediction**

Decision trees are **interpretable, intuitive, and powerful**—and now you know **exactly how they’re built**! 🌳✨

# Optional
# 🔍 How Is the Split Value (Threshold) Determined?

In a decision tree, the **split value** (or **threshold**) for a numerical feature—like `Credit_Score ≥ ?`—is **not guessed**.  
It is **systematically computed** from the data to find the **best possible separation** between classes.

Let’s see **exactly how it works**.

---

## 📊 Step 1: Use Actual Data Values

Suppose in the current node, the `Credit_Score` values are:
[580, 600, 620, 650, 670, 690, 720, 750]

These are **real samples** from your dataset.

---

## 📐 Step 2: Generate Candidate Thresholds

The algorithm **sorts** these values and computes **midpoints** between every **consecutive pair**:

| Between          | Midpoint (Candidate Threshold) |
|------------------|-------------------------------|
| 580 and 600      | (580 + 600) / 2 = **590**     |
| 600 and 620      | (600 + 620) / 2 = **610**     |
| 620 and 650      | (620 + 650) / 2 = **635**     |
| 650 and 670      | (650 + 670) / 2 = **660**     |
| **670 and 690**  | **(670 + 690) / 2 = 680** ✅  |
| 690 and 720      | (690 + 720) / 2 = **705**     |
| 720 and 750      | (720 + 750) / 2 = **735**     |

> 💡 **Why midpoints?**  
> Any threshold **between 670 and 690** (e.g., 675, 680, 689) produces the **exact same split**:  
> - Left: scores ≤ 670  
> - Right: scores ≥ 690  
> So we test **only the midpoint (680)** as a representative.

---

## ⚖️ Step 3: Evaluate Each Candidate Using Gini Impurity

For **each candidate**, the algorithm:
1. Splits the data into **left** and **right** subsets
2. Computes **Gini impurity** for each subset
3. Calculates the **weighted average Gini**
4. Computes **Gini Gain** = `Gini(parent) − Weighted Gini(children)`

### Example: Compare Two Candidates

#### Candidate: Threshold = **660**
- **Left** (<660): [580, 600, 620, 650] → Labels: [No, No, No, Yes]  
  → Gini = 0.375  
- **Right** (≥660): [670, 690, 720, 750] → Labels: [No, Yes, Yes, Yes]  
  → Gini = 0.375  
- **Weighted Gini** = (4/8)(0.375) + (4/8)(0.375) = **0.375**  
- **Gini Gain** = 0.4688 − 0.375 = **0.0938**

#### Candidate: Threshold = **680** ← **Best!**
- **Left** (<680): [580, 600, 620, 650, 670] → Labels: [No, No, No, Yes, No]  
  → Gini = 0.32  
- **Right** (≥680): [690, 720, 750] → Labels: [Yes, Yes, Yes]  
  → Gini = **0.0** (pure!)  
- **Weighted Gini** = (5/8)(0.32) + (3/8)(0) = **0.20**  
- **Gini Gain** = 0.4688 − 0.20 = **0.2688** ✅ **Highest!**

---

## ✅ Step 4: Choose the Best Threshold

The algorithm selects the threshold with the **maximum Gini Gain**.

In this case:  
> **Best split: `Credit_Score < 680`**

This creates:
- A **pure right node** (all "Approve = Yes")
- The **greatest reduction in impurity**

---

## 🌟 Key Takeaways

- The split value (**680**) is **derived from your data**, not predefined.
- Only **midpoints between existing values** are tested—this is **efficient and exhaustive**.
- The "best" split is the one that **maximizes class separation** (via Gini Gain or Information Gain).
- This process repeats **recursively** at every node during tree construction.

> 🌳 **Decision trees learn optimal thresholds directly from your data—no assumptions, just math!**

In [ ]:
import numpy as np
import pandas as pd

# Sample data
df = pd.DataFrame({
    'Credit_Score': [580, 600, 620, 650, 670, 690, 720, 750],
    'Approve':      [0,    0,    0,    1,    0,    1,    1,    1]
})

feature = 'Credit_Score'
target = 'Approve'

# Get unique sorted values
values = np.sort(df[feature].unique())
print("Unique Credit Scores:", values)

# Generate candidate thresholds (midpoints)
thresholds = []
for i in range(len(values) - 1):
    thr = (values[i] + values[i+1]) / 2
    thresholds.append(thr)

print("\nCandidate thresholds:")
for t in thresholds:
    print(f"  {t}")

# Evaluate one threshold
t = 680
left_mask = df[feature] < t
gini_left = 1 - (df[target][left_mask].value_counts(normalize=True) ** 2).sum()
gini_right = 1 - (df[target][~left_mask].value_counts(normalize=True) ** 2).sum()

print(f"\nAt threshold = {t}:")
print(f"  Left Gini:  {gini_left:.3f}")
print(f"  Right Gini: {gini_right:.3f}")

Unique Credit Scores: [580 600 620 650 670 690 720 750]

Candidate thresholds:
  590.0
  610.0
  635.0
  660.0
  680.0
  705.0
  735.0

At threshold = 680:
  Left Gini:  0.320
  Right Gini: 0.000
